# M16 (idea stage) · 00 — System specs and GPU training throughput

**Status:** exploratory. M16 (our own LLM) is not in `docs/development-plan.md` yet; this notebook feeds the discussion with the owner before planning.

Questions:
1. What hardware do we actually have (GPU, VRAM, CPU, RAM, disk)?
2. What matmul throughput does the RTX 5070 reach in TF32 / BF16 / FP16 / FP8?
3. How many tokens/s does a real training step reach for a Llama-style decoder at ~124M and ~350M parameters (bf16 autocast, `torch.compile`, fused AdamW, SDPA attention)?
4. Given that, how long would compute-optimal pretraining runs take?

Run from WSL with `experiments/.venv` (torch 2.8.0+cu128). Stop the worker first so nothing else holds VRAM.

In [1]:
import os, platform, shutil, subprocess, time, gc, math
import torch, torch.nn as nn, torch.nn.functional as F

def sh(cmd):
    return subprocess.run(['bash', '-c', cmd], capture_output=True, text=True).stdout.strip()

print('OS      :', platform.platform())
print('CPU     :', sh("lscpu | grep 'Model name' | sed 's/.*: *//'"), '|', os.cpu_count(), 'logical cores')
print(f"RAM     : {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30:.1f} GiB visible to WSL")
print(f"Disk    : {shutil.disk_usage(os.path.expanduser('~')).free / 2**30:.0f} GiB free on WSL ext4 (~)")
print('torch   :', torch.__version__, '| CUDA', torch.version.cuda, '| cuDNN', torch.backends.cudnn.version())
p = torch.cuda.get_device_properties(0)
print(f'GPU     : {p.name} | {p.total_memory / 2**30:.1f} GiB | sm_{p.major}{p.minor} | {p.multi_processor_count} SMs')
print(sh('nvidia-smi --query-gpu=driver_version,memory.used,power.limit,clocks.max.sm --format=csv'))

OS      : Linux-6.18.54.1-microsoft-standard-WSL2-x86_64-with-glibc2.39
CPU     : AMD Ryzen 7 7800X3D 8-Core Processor | 16 logical cores
RAM     : 23.0 GiB visible to WSL
Disk    : 836 GiB free on WSL ext4 (~)
torch   : 2.8.0+cu128 | CUDA 12.8 | cuDNN 91002
GPU     : NVIDIA GeForce RTX 5070 | 11.9 GiB | sm_120 | 48 SMs
driver_version, memory.used [MiB], power.limit [W], clocks.max.sm [MHz]
617.14, 547 MiB, 250.00 W, 3090 MHz


## Matmul throughput
Square GEMMs, 50 timed iterations after warm-up. This is the ceiling a training step can approach, not what it reaches.

In [2]:
def timeit(fn, iters=50, warm=5):
    for _ in range(warm): fn()
    torch.cuda.synchronize(); t = time.perf_counter()
    for _ in range(iters): fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t) / iters

torch.backends.cuda.matmul.allow_tf32 = True
mm_rows = []
for name, dtype in [('tf32', torch.float32), ('bf16', torch.bfloat16), ('fp16', torch.float16)]:
    for n in (2048, 4096, 8192):
        a = torch.randn(n, n, device='cuda', dtype=dtype); b = torch.randn(n, n, device='cuda', dtype=dtype)
        mm_rows.append((name, n, 2 * n**3 / timeit(lambda: a @ b) / 1e12))
        del a, b

try:  # FP8 e4m3 via torch._scaled_mm (second operand must be column-major)
    for n in (4096, 8192):
        a = torch.randn(n, n, device='cuda').to(torch.float8_e4m3fn)
        b = torch.randn(n, n, device='cuda').to(torch.float8_e4m3fn).t()
        s = torch.tensor(1.0, device='cuda')
        mm_rows.append(('fp8', n, 2 * n**3 / timeit(lambda: torch._scaled_mm(a, b, scale_a=s, scale_b=s, out_dtype=torch.bfloat16)) / 1e12))
        del a, b
except Exception as e:
    print('FP8 not available:', type(e).__name__, e)

for name, n, tf in mm_rows:
    print(f'{name:5s} n={n:5d}  {tf:6.1f} TFLOPS')
BF16_PEAK = max(tf for name, _, tf in mm_rows if name == 'bf16')
torch.cuda.empty_cache()

tf32  n= 2048    28.5 TFLOPS
tf32  n= 4096    31.7 TFLOPS
tf32  n= 8192    31.9 TFLOPS
bf16  n= 2048    51.4 TFLOPS
bf16  n= 4096    57.6 TFLOPS
bf16  n= 8192    62.4 TFLOPS
fp16  n= 2048    59.1 TFLOPS
fp16  n= 4096    61.5 TFLOPS
fp16  n= 8192    59.6 TFLOPS
fp8   n= 4096   113.9 TFLOPS
fp8   n= 8192   122.8 TFLOPS


## Real training step: Llama-style decoder
RMSNorm, RoPE, SwiGLU, causal SDPA (flash), tied embeddings, vocab 32,768, sequence 2,048. bf16 autocast with fp32 master weights and fused AdamW (16 bytes/param of weight + optimizer state). Random tokens: throughput does not depend on the data.

FLOPs/token = 6·(non-embedding params + d·V) + 12·L·d·T (attention scores), as in the nanoGPT/nanochat MFU estimate. Micro-batch is the largest that fits, halving on OOM.

**Run 1 (2026-11-14) was invalid.** Peak memory was 15.4 / 21.2 / 22.0 GiB on an 11.9 GiB card. The Windows driver silently spilled into system RAM (sysmem fallback), giving 3,911 / 861 / 588 tok/s and an MFU of 3–5 %. Since run 2 the allocator is capped at 92 % of VRAM with `set_per_process_memory_fraction`, so a spill raises OOM instead of running slowly, and smaller micro-batches are tried. Any real training run needs the same cap (or "Prefer No Sysmem Fallback" in the NVIDIA control panel).

In [3]:
V, T = 32768, 2048
MEM_FRACTION = 0.92
torch.cuda.set_per_process_memory_fraction(MEM_FRACTION)  # never spill to system RAM (run 1)
VRAM_CAP_GIB = MEM_FRACTION * torch.cuda.get_device_properties(0).total_memory / 2**30

def is_oom(e):
    return isinstance(e, torch.OutOfMemoryError) or 'out of memory' in str(e).lower()

def rope_tables(T, hd, base=10000.0):
    inv = 1.0 / base ** (torch.arange(0, hd, 2, device='cuda').float() / hd)
    f = torch.outer(torch.arange(T, device='cuda').float(), inv)
    return f.cos().bfloat16(), f.sin().bfloat16()

def rope(x, cos, sin):
    x1, x2 = x.chunk(2, -1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1)

class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h = h
        self.n1, self.n2 = nn.RMSNorm(d), nn.RMSNorm(d)
        self.qkv, self.o = nn.Linear(d, 3 * d, bias=False), nn.Linear(d, d, bias=False)
        hid = int(8 * d / 3) // 64 * 64
        self.w12, self.w3 = nn.Linear(d, 2 * hid, bias=False), nn.Linear(hid, d, bias=False)

    def forward(self, x, cos, sin):
        B, T, D = x.shape
        q, k, v = self.qkv(self.n1(x)).view(B, T, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        y = F.scaled_dot_product_attention(rope(q, cos, sin), rope(k, cos, sin), v, is_causal=True)
        x = x + self.o(y.transpose(1, 2).reshape(B, T, D))
        a, b = self.w12(self.n2(x)).chunk(2, -1)
        return x + self.w3(F.silu(a) * b)

class GPT(nn.Module):
    def __init__(self, d, L, h):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.blocks = nn.ModuleList(Block(d, h) for _ in range(L))
        self.norm = nn.RMSNorm(d)
        self.cos, self.sin = rope_tables(T, d // h)

    def forward(self, idx, tgt):
        x = self.emb(idx)
        for blk in self.blocks:
            x = blk(x, self.cos, self.sin)
        logits = F.linear(self.norm(x), self.emb.weight)  # tied head
        return F.cross_entropy(logits.float().view(-1, V), tgt.reshape(-1))

def bench_train(name, d, L, h, batches, steps=10):
    for B in batches:
        model = opt = step = None
        try:
            torch._dynamo.reset(); gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            model = GPT(d, L, h).cuda()
            n_total = sum(p.numel() for p in model.parameters())
            n_nonemb = n_total - V * d
            opt = torch.optim.AdamW(model.parameters(), lr=1e-4, fused=True)
            cm = torch.compile(model)
            x = torch.randint(0, V, (B, T + 1), device='cuda')
            def step():
                with torch.autocast('cuda', dtype=torch.bfloat16):
                    loss = cm(x[:, :-1], x[:, 1:])
                loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
            t0 = time.perf_counter(); step(); torch.cuda.synchronize(); compile_s = time.perf_counter() - t0
            dt = timeit(step, iters=steps, warm=2)
            peak = torch.cuda.max_memory_allocated() / 2**30
            assert peak <= VRAM_CAP_GIB + 0.1, f'peak {peak:.1f} GiB above the cap: spill'
            tok_s = B * T / dt
            flops_tok = 6 * (n_nonemb + d * V) + 12 * L * d * T
            return dict(name=name, params_M=n_total / 1e6, nonemb_M=n_nonemb / 1e6, B=B, tok_s=tok_s,
                        tflops=tok_s * flops_tok / 1e12, mfu=tok_s * flops_tok / 1e12 / BF16_PEAK,
                        peak_GiB=peak, compile_s=compile_s, flops_tok=flops_tok)
        except Exception as e:
            if not is_oom(e):
                raise
            print(f'{name}: B={B} OOM, halving')
        finally:
            del model, opt, step
            gc.collect(); torch.cuda.empty_cache()
    return None

In [4]:
configs = [
    ('~124M (d768 L12)', 768, 12, 12, (8, 4, 2, 1)),
    ('~350M (d1024 L24)', 1024, 24, 16, (4, 2, 1)),
    ('~700M (d1536 L24)', 1536, 24, 12, (2, 1)),  # 16 B/param x 730M = 11.7 GB of state alone: expected not to fit
]
results = []
for cfg in configs:
    r = bench_train(*cfg)
    if r is None:
        print(f'{cfg[0]}: does not fit with fp32 AdamW even at B=1')
        continue
    results.append(r)
    print(f"{r['name']:20s} params {r['params_M']:6.0f}M  B={r['B']:2d}  {r['tok_s']:8,.0f} tok/s  "
          f"{r['tflops']:5.1f} TFLOPS  MFU {r['mfu']:.0%}  peak {r['peak_GiB']:.1f} GiB  compile {r['compile_s']:.0f}s")

W1114 13:22:04.055000 1239 torch/_inductor/utils.py:1436] [0/0] Not enough SMs to use max_autotune_gemm mode


~124M (d768 L12)     params    110M  B= 8    60,577 tok/s   53.7 TFLOPS  MFU 86%  peak 8.5 GiB  compile 16s


~350M (d1024 L24): B=4 OOM, halving


~350M (d1024 L24)    params    332M  B= 2    18,629 tok/s   48.4 TFLOPS  MFU 78%  peak 8.7 GiB  compile 21s


~700M (d1536 L24): B=2 OOM, halving


~700M (d1536 L24): B=1 OOM, halving


~700M (d1536 L24): does not fit with fp32 AdamW even at B=1


## What this means in wall-clock time
Compute-optimal (Chinchilla) is ~20 tokens/param; modern small models are trained far past that (SmolLM2-360M: ~11T tokens), so 100 tokens/param is shown too. nanochat's d20 speedrun ("GPT-2 grade", ~4e19 FLOPs on 8×H100 in ~4 h) is the reference point for "a model you can chat with".

In [5]:
print(f"{'model':20s} {'tokens':>8s} {'days':>7s}")
for r in results:
    for mult in (20, 100):
        tokens = mult * r['params_M'] * 1e6
        print(f"{r['name']:20s} {tokens / 1e9:6.1f}B {tokens / r['tok_s'] / 86400:7.1f}   ({mult} tok/param)")
if results:
    best_tflops = max(r['tflops'] for r in results)
    print(f"\nnanochat d20 budget (4e19 FLOPs) at {best_tflops:.1f} TFLOPS sustained: {4e19 / (best_tflops * 1e12) / 86400:.1f} days")

model                  tokens    days
~124M (d768 L12)        2.2B     0.4   (20 tok/param)
~124M (d768 L12)       11.0B     2.1   (100 tok/param)
~350M (d1024 L24)       6.6B     4.1   (20 tok/param)
~350M (d1024 L24)      33.2B    20.7   (100 tok/param)

nanochat d20 budget (4e19 FLOPs) at 53.7 TFLOPS sustained: 8.6 days
